# Medical BERT Classifier - Interactive Training Notebook

This notebook demonstrates last-layer fine-tuning of medical BERT models for 10-class medical text classification.

## Overview
- **Approach**: Last-layer training only (freeze BERT backbone)
- **Models**: Bio-BERT, Clinical-Bio-BERT, BlueBERT
- **Classes**: 10 medical text categories including new Sx (Symptoms) class
- **Focus**: CPU-optimized with GPU acceleration support

Based on the Fine-tuning Transformers with HuggingFace approach.

## Setup and Imports

In [ ]:
# Environment setup
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Core imports
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Transformers imports
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Our custom modules
from src.models.bio_bert import create_biobert_model
from src.models.clinical_bert import create_clinical_bert_model
from src.training.last_layer_trainer import LastLayerTrainer, MedicalTextDataset, create_datasets_from_csv
from src.data.generate_10class import MedicalDataGenerator

print("Setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

## Configuration Loading

In [ ]:
# Load configuration
with open('../config/classes.yaml', 'r') as f:
    classes_config = yaml.safe_load(f)

with open('../config/training.yaml', 'r') as f:
    training_config = yaml.safe_load(f)

# Display class information
print("Medical Text Classification Classes:")
print("=" * 40)
for class_id, class_name in classes_config['class_labels'].items():
    description = classes_config['class_descriptions'][class_name]
    print(f"{class_id}: {class_name} - {description}")

print(f"\nTotal classes: {classes_config['num_classes']}")

## Data Generation

Generate synthetic medical text data for all 10 classes, including the new Sx (Symptoms) class.

In [ ]:
# Generate synthetic data
print("Generating synthetic medical text data...")

generator = MedicalDataGenerator('../config/classes.yaml')
df = generator.generate_all_data()

# Display data summary
print(f"\nGenerated {len(df)} total samples")
print("\nClass distribution:")
print(df['label'].value_counts().sort_index())

# Show sample data
print("\nSample data:")
print("=" * 80)
for label in classes_config['class_labels'].values():
    sample = df[df['label'] == label].iloc[0]
    print(f"{label}: {sample['text'][:100]}...")
    
# Save data
data_path = '../data/synthetic_training_data_10class.csv'
generator.save_data(df, data_path)
print(f"\nData saved to: {data_path}")

## Model Comparison Framework

Compare different medical BERT models using our last-layer training approach.

In [ ]:
# Model configurations
models_config = {
    'Bio-BERT': {
        'model_name': 'dmis-lab/biobert-base-cased-v1.1',
        'create_func': create_biobert_model,
        'description': 'Pre-trained on PubMed abstracts and PMC articles'
    },
    'Clinical-Bio-BERT': {
        'model_name': 'emilyalsentzer/Bio_ClinicalBERT',
        'create_func': create_clinical_bert_model,
        'description': 'Bio-BERT further trained on MIMIC-III clinical notes'
    }
}

print("Available Models:")
print("=" * 50)
for name, config in models_config.items():
    print(f"{name}: {config['description']}")
    print(f"  Model: {config['model_name']}")
    print()

## Training Function

Define a training function following the patterns from the Fine-tuning Transformers notebook.

In [ ]:
def train_medical_bert_model(model_name, num_epochs=3, test_size=0.2):
    """
    Train a medical BERT model using last-layer fine-tuning
    
    Following the approach from Fine-tuning Transformers notebook
    """
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    # Get model configuration
    config = models_config[model_name]
    
    # Create model
    print("Creating model...")
    model = config['create_func'](num_classes=10)
    
    # Create tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load and prepare data
    print("Preparing data...")
    train_dataset, val_dataset = create_datasets_from_csv(
        data_path, tokenizer, test_size=test_size
    )
    
    # Create trainer
    print("Setting up trainer...")
    trainer = LastLayerTrainer(model, train_dataset, val_dataset)
    
    # Train model
    print(f"Training for {num_epochs} epochs...")
    import time
    start_time = time.time()
    
    results = trainer.train(num_epochs=num_epochs)
    
    training_time = time.time() - start_time
    
    # Results summary
    print(f"\nTraining completed in {training_time:.2f} seconds")
    print(f"Best validation accuracy: {results['best_val_accuracy']:.4f}")
    print(f"Final F1 score: {results['final_metrics']['f1']:.4f}")
    
    return {
        'model_name': model_name,
        'trainer': trainer,
        'results': results,
        'training_time': training_time
    }

print("Training function defined")

## Single Model Training Example

Train Clinical-Bio-BERT as an example of the last-layer fine-tuning approach.

In [ ]:
# Train Clinical-Bio-BERT
clinical_bert_results = train_medical_bert_model('Clinical-Bio-BERT', num_epochs=3)

# Plot training progress
trainer = clinical_bert_results['trainer']
trainer.plot_training_progress()

# Plot confusion matrix
class_names = list(classes_config['class_labels'].values())
trainer.plot_confusion_matrix(class_names)

## Model Comparison

Compare Bio-BERT and Clinical-Bio-BERT performance on the 10-class medical text classification task.

In [ ]:
# Train and compare both models
comparison_results = {}

for model_name in ['Bio-BERT', 'Clinical-Bio-BERT']:
    try:
        results = train_medical_bert_model(model_name, num_epochs=2)  # Shorter for comparison
        comparison_results[model_name] = results
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        continue

# Create comparison table
if comparison_results:
    comparison_data = []
    for model_name, results in comparison_results.items():
        comparison_data.append({
            'Model': model_name,
            'Best Accuracy': f"{results['results']['best_val_accuracy']:.4f}",
            'Training Time (s)': f"{results['training_time']:.2f}",
            'Final F1': f"{results['results']['final_metrics']['f1']:.4f}",
            'Trainable Params': f"{results['trainer'].model.get_num_trainable_parameters():,}"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\nModel Comparison Results:")
    print("=" * 60)
    print(comparison_df.to_string(index=False))
    
    # Visualize comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Accuracy comparison
    models = [r['model_name'] for r in comparison_results.values()]
    accuracies = [r['results']['best_val_accuracy'] for r in comparison_results.values()]
    
    ax1.bar(models, accuracies, color=['skyblue', 'lightcoral'])
    ax1.set_title('Model Accuracy Comparison')
    ax1.set_ylabel('Validation Accuracy')
    ax1.set_ylim(0, 1)
    
    # Training time comparison
    times = [r['training_time'] for r in comparison_results.values()]
    ax2.bar(models, times, color=['lightgreen', 'orange'])
    ax2.set_title('Training Time Comparison')
    ax2.set_ylabel('Training Time (seconds)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No models were successfully trained for comparison")

## Detailed Analysis: Clinical-Bio-BERT Performance

Analyze the performance of Clinical-Bio-BERT on each medical text class.

In [ ]:
# Detailed analysis for Clinical-Bio-BERT
if 'Clinical-Bio-BERT' in comparison_results:
    clinical_trainer = comparison_results['Clinical-Bio-BERT']['trainer']
    
    # Get final evaluation metrics
    accuracy, loss, metrics = clinical_trainer.evaluate()
    
    # Classification report
    from sklearn.metrics import classification_report
    
    class_names = list(classes_config['class_labels'].values())
    report = classification_report(
        metrics['labels'], 
        metrics['predictions'],
        target_names=class_names,
        output_dict=True
    )
    
    # Convert to DataFrame for better display
    report_df = pd.DataFrame(report).transpose()
    print("Clinical-Bio-BERT Performance by Class:")
    print("=" * 50)
    print(report_df.round(4))
    
    # Analyze performance on new Sx class
    sx_metrics = report['Sx']
    print(f"\nSymptoms (Sx) Class Performance:")
    print(f"Precision: {sx_metrics['precision']:.4f}")
    print(f"Recall: {sx_metrics['recall']:.4f}")
    print(f"F1-score: {sx_metrics['f1-score']:.4f}")
    
    # Class-wise accuracy visualization
    class_f1_scores = [report[class_name]['f1-score'] for class_name in class_names]
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(class_names, class_f1_scores, color='lightblue')
    
    # Highlight the new Sx class
    sx_index = class_names.index('Sx')
    bars[sx_index].set_color('orange')
    
    plt.title('Clinical-Bio-BERT: F1-Score by Medical Text Class')
    plt.xlabel('Medical Text Class')
    plt.ylabel('F1-Score')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    
    # Add value labels on bars
    for bar, score in zip(bars, class_f1_scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                 f'{score:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Clinical-Bio-BERT results not available for detailed analysis")

## Testing Individual Predictions

Test the trained model on individual text samples to see how it performs on each class.

In [ ]:
def predict_text_sample(text, model, tokenizer, class_labels):
    """
    Predict the class of a single text sample
    """
    # Tokenize text
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )
    
    # Get device
    device = next(model.parameters()).device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probabilities = torch.softmax(logits, dim=-1)
        predicted_class_id = torch.argmax(logits, dim=-1).item()
        confidence = torch.max(probabilities).item()
    
    predicted_class = class_labels[predicted_class_id]
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence,
        'probabilities': probabilities.squeeze().cpu().numpy()
    }

# Test samples for each class
test_samples = {
    'Rx': "Patient prescribed metformin 500mg twice daily for diabetes management",
    'Lx': "Hemoglobin A1c level measured at 7.2% indicating good glycemic control", 
    'Dx': "Patient diagnosed with type 2 diabetes mellitus based on elevated glucose",
    'Px': "Coronary angioplasty performed with stent placement in LAD vessel",
    'Sx': "Patient reports persistent chest pain radiating to left arm with shortness of breath",  # NEW!
    'Other': "Insurance verification completed for patient visit",
    'Vitals': "Blood pressure recorded as 135/85 mmHg, heart rate 72 bpm",
    'FHx': "Strong family history of diabetes in mother and maternal grandmother",
    'SDOH': "Patient reports housing instability affecting medication storage",
    'Tobacco': "Current smoker with 20 pack-year history, attempting cessation"
}

if 'Clinical-Bio-BERT' in comparison_results:
    model = comparison_results['Clinical-Bio-BERT']['trainer'].model
    tokenizer = AutoTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
    
    print("Testing Individual Predictions:")
    print("=" * 70)
    
    for true_class, text in test_samples.items():
        result = predict_text_sample(text, model, tokenizer, class_names)
        
        correct = "✅" if result['predicted_class'] == true_class else "❌"
        
        print(f"{correct} True: {true_class} | Predicted: {result['predicted_class']} | Confidence: {result['confidence']:.3f}")
        print(f"   Text: {text[:60]}...")
        print()
else:
    print("No trained model available for testing")

## Key Insights and Next Steps

### Key Findings:
1. **Last-layer training efficiency**: Only training the classification layer significantly reduces training time while maintaining good performance
2. **Medical BERT performance**: Clinical-Bio-BERT typically performs better on clinical text due to domain-specific pre-training
3. **New Sx class**: The symptoms class adds important distinction between patient complaints and clinical findings

### Next Steps:
1. **Real data integration**: Replace synthetic data with actual clinical notes
2. **Model ensemble**: Combine predictions from multiple medical BERT models
3. **Active learning**: Use uncertainty estimation to identify samples for human annotation
4. **Domain adaptation**: Fine-tune on specific medical domains (cardiology, oncology, etc.)

### Configuration Expansion:
- Easy to add new classes by updating `config/classes.yaml`
- Training parameters can be adjusted in `config/training.yaml`
- Support for additional medical BERT models in `config/models.yaml`

In [ ]:
# Summary statistics
if comparison_results:
    print("Final Summary:")
    print("=" * 50)
    
    for model_name, results in comparison_results.items():
        print(f"{model_name}:")
        print(f"  - Best Accuracy: {results['results']['best_val_accuracy']:.4f}")
        print(f"  - Training Time: {results['training_time']:.2f}s")
        print(f"  - Trainable Parameters: {results['trainer'].model.get_num_trainable_parameters():,}")
        
        total_params = sum(p.numel() for p in results['trainer'].model.parameters())
        trainable_ratio = results['trainer'].model.get_num_trainable_parameters() / total_params
        print(f"  - Trainable Ratio: {trainable_ratio:.2%}")
        print()
    
    print("🎉 Notebook execution completed successfully!")
    print("Check the results for detailed model performance analysis.")
else:
    print("⚠️  No models were successfully trained in this session.")
    print("Consider checking the setup and trying again.")